# ResNet9 CNN Examples

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "python").exists() and (ROOT.parent / "python").exists():
    ROOT = ROOT.parent
EXAMPLE_DIR = ROOT / "example_resnet"

PYTHON_DIR = ROOT / "python"
BUILD_DIR = ROOT / "build"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))
if BUILD_DIR.exists() and str(BUILD_DIR) not in sys.path:
    sys.path.insert(0, str(BUILD_DIR))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import backend_ndarray as nd  # noqa: E402
import nn  # noqa: E402
from optim import AdamW  # noqa: E402
from example_resnet.resnet import ResNet9, epoch, smoke_train_step, cifar10_loader  # noqa: E402

Pick a backend. If the Metal extension is built and importable, `default_device()` will choose it; otherwise it falls back to NumPy.

In [2]:
device = nd.default_device()
device

metal()

First run a synthetic single-step smoke test. This needs no dataset and is the fastest check that the model can forward, backward, and update parameters.

In [3]:
smoke_train_step(device=device, batch_size=2, seed=0)

{'device': 'metal',
 'logits_shape': (2, 10),
 'loss': 3.8681840896606445,
 'accuracy': 0.0,
 'parameters': 36}

CIFAR-10

In [4]:
from example_resnet import ensure_cifar10

data_dir = ensure_cifar10(EXAMPLE_DIR)

batch_size = 128
epochs = 5
augment = True
data_dir

PosixPath('/Users/mark/Desktop/github/metallic/example_resnet/cifar-10-batches-py')

In [5]:
train_loader = cifar10_loader(
    data_dir,
    train=True,
    batch_size=batch_size,
    device=device,
    augment=augment,
)
x, y = next(iter(train_loader))
print(x.shape, y.shape)

model = ResNet9(device=device)
opt = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
history = []
for epoch_idx in range(epochs):
    train_acc, train_loss = epoch(
        train_loader, model, loss_fn=nn.SoftmaxLoss(), opt=opt
    )
    row = {
        "epoch": epoch_idx + 1,
        "train_accuracy": train_acc,
        "train_loss": train_loss,
    }
    history.append(row)
    print(row)
history

/Users/mark/Desktop/github/metallic/example_resnet/data.py:155: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  batch = pickle.load(file, encoding="bytes")


(128, 3, 32, 32) (128,)
{'epoch': 1, 'train_accuracy': 0.32542, 'train_loss': 1.848620541191101}
{'epoch': 2, 'train_accuracy': 0.42174, 'train_loss': 1.5889708811187744}
{'epoch': 3, 'train_accuracy': 0.4559, 'train_loss': 1.4850030994033814}
{'epoch': 4, 'train_accuracy': 0.48404, 'train_loss': 1.4185216818618775}
{'epoch': 5, 'train_accuracy': 0.50328, 'train_loss': 1.375554825553894}


[{'epoch': 1, 'train_accuracy': 0.32542, 'train_loss': 1.848620541191101},
 {'epoch': 2, 'train_accuracy': 0.42174, 'train_loss': 1.5889708811187744},
 {'epoch': 3, 'train_accuracy': 0.4559, 'train_loss': 1.4850030994033814},
 {'epoch': 4, 'train_accuracy': 0.48404, 'train_loss': 1.4185216818618775},
 {'epoch': 5, 'train_accuracy': 0.50328, 'train_loss': 1.375554825553894}]